In [2]:
from utils import * 
import subprocess
import json
from Bio.Align import PairwiseAligner 
import itertools
from dssp import DSSPFile
from blast import BLASTFileJSON
import requests

%load_ext autoreload 
%autoreload 2

In [22]:
genes_df = pd.read_csv('../data/genes.csv', index_col=0)
genes_df = genes_df[(genes_df.cluster_id == 8)].copy()
print('\n'.join(genes_df.index))
FASTAFile.from_df(genes_df).write('../data/genes/cluster_8.0.faa')

print('\nMaximum length of a cluster_8 gene:', genes_df.seq.apply(len).max())
print('Minimum length of a cluster_8 gene:', genes_df.seq.apply(len).min())


orfm.bz_3.1_77
orfm.bz_4.1_146
orfm.bz_5.1_120
orfm.bz_7.1_133
orfm.bz_9.1_86

Maximum length of a cluster_8 gene: 642
Minimum length of a cluster_8 gene: 538


In [ ]:
# ! blastp -db ../data/genes/blast/db/genes/genes -outfmt "6 qseqid sseqid pident length qlen slen qstart qend sstart send evalue bitscore" -query ../data/genes/cluster_8.0.faa -out ../data/genes/blast/cluster_8.0-genes.tsv
# ! blastp -db ../data/genes/blast/db/genes/genes -outfmt "6 qseqid sseqid pident length qlen slen qstart qend sstart send evalue bitscore" -query ../data/genes/cluster_8.1.faa -out ../data/genes/blast/cluster_8.1-genes.tsv
BLASTP_DATABASE_PATH = '../data/genes/blast/db/genes/genes'
BLASTP_QUERY_PATH = '../data/genes/cluster_8.0.faa'
BLASTP_OUTPUT_PATH = '../data/genes/blast/cluster_7.0-genes.tsv'
BLASTP_FIELDS = 'qseqid sseqid pident length qlen slen qstart qend sstart send evalue bitscore'

cmd = f'blastp -db {BLASTP_DATABASE_PATH} -outfmt "6 {BLASTP_FIELDS}" -query {BLASTP_QUERY_PATH} -out {BLASTP_OUTPUT_PATH}'
print(cmd)

blastp -db ../data/genes/blast/db/genes/genes -outfmt "6 qseqid sseqid pident length qlen slen qstart qend sstart send evalue bitscore" -query ../data/genes/cluster_8.0.faa -out ../data/genes/blast/cluster_7.0-genes.tsv


In [35]:
def load_blast(path:str):
    
    blast_df = pd.read_csv(path, sep='\t', names=BLASTP_FIELDS.split())
    blast_df['subject_genome_id'] = blast_df.sseqid.apply(get_genome_id)
    blast_df['query_genome_id'] = blast_df.qseqid.apply(get_genome_id)
    blast_df['subject_confirmed'] = blast_df.sseqid.isin(genes_df.index)
    blast_df['false_positive'] = (blast_df.subject_genome_id == blast_df.query_genome_id) & (blast_df.qseqid != blast_df.sseqid) # These are not valid, assuming one copy per genome.
    blast_df = blast_df.sort_values('bitscore', ascending=False) # Sort so that dropping duplicates keeps the best hits.

    # print('load_blast: Maximum bit score of false positive hits:', blast_df[blast_df.false_positive].bitscore.max())

    filters = dict()
    filters['self_alignments'] = blast_df.qseqid == blast_df.sseqid
    filters['query_and_subject_in_same_genome'] = blast_df.subject_genome_id == blast_df.query_genome_id
    filters['bz_12_hit'] = blast_df.subject_genome_id == 'bz_12'
    filters['duplicated_query_subject_pair'] = blast_df.duplicated(['qseqid', 'sseqid'], keep='first')

    blast_df = apply_filters(filters, blast_df)
    
    return blast_df

In [27]:
# Searching for genes proximal to cluster_5 is not producing results as clearly as it did for finding cluster_5 genes. 
# Instead, I am trying a BLASTp search to see if local sequence aligment is more sensitive than the MMseqs or Foldseek clustering. 

blast_df = load_blast('../data/genes/blast/cluster_8.0-genes.tsv')

print('Number of genomes with BLAST hits:', blast_df.subject_genome_id.nunique())
print('Number of genomes with strong BLAST hits:', blast_df[blast_df.bitscore > 45].subject_genome_id.nunique())
print('Number of genomes with confirmed cluster_8 proteins:', genes_df.genome_id.nunique())

cluster_8_gene_ids = genes_df.index.tolist() + blast_df[blast_df.bitscore > 45].sseqid.tolist()
cluster_8_gene_ids = np.unique(cluster_8_gene_ids).tolist()

genes_df = pd.read_csv('../data/genes.csv', index_col=0).loc[cluster_8_gene_ids].copy()
FASTAFile.from_df(genes_df).write('../data/genes/cluster_8.1.faa')

# BLAST iteration number 2...  

blast_df = load_blast('../data/genes/blast/cluster_8.1-genes.tsv')

print('Number of genomes with BLAST hits:', blast_df.subject_genome_id.nunique())
print('Number of genomes with strong BLAST hits:', blast_df[blast_df.bitscore > 30].subject_genome_id.nunique())
print('Number of genomes with confirmed cluster_8 proteins:', genes_df.genome_id.nunique())

cluster_8_gene_ids = genes_df.index.tolist() + blast_df[blast_df.bitscore > 30].sseqid.tolist()
cluster_8_gene_ids = np.unique(cluster_8_gene_ids).tolist()



load_blast: Maximum bit score of false positive hits: 24.3
apply_filters: 5 entries removed by self_alignments.
apply_filters: 10 entries removed by query_and_subject_in_same_genome.
apply_filters: 3 entries removed by bz_12_hit.
apply_filters: 10 entries removed by duplicated_query_subject_pair.
load_blast: Minimum bit score between confirmed cluster_7 proteins: 69.7
load_blast: Maximum bit score between confirmed cluster_7 proteins: 163.0
load_blast: Mean bit score between confirmed cluster_7 proteins: 107.34
Number of genomes with BLAST hits: 11
Number of genomes with strong BLAST hits: 7
Number of genomes with confirmed cluster_8 proteins: 5
load_blast: Maximum bit score of false positive hits: 24.3
apply_filters: 7 entries removed by self_alignments.
apply_filters: 14 entries removed by query_and_subject_in_same_genome.
apply_filters: 4 entries removed by bz_12_hit.
apply_filters: 11 entries removed by duplicated_query_subject_pair.
load_blast: Minimum bit score between confirmed 

In [34]:
df = pd.read_csv('../data/genes.csv', index_col=0)
df = df[(df.length < 700) & (df.length > 500)].copy()
df['confirmed_cluster_8'] = df.index.isin(cluster_8_gene_ids)

df[['length', 'start', 'confirmed_cluster_8', 'strand', 'prodigal_gene_id']].sort_values('confirmed_cluster_8')

# genes_df.genome_id.unique()

,length,start,confirmed_cluster_8,strand,prodigal_gene_id
gene_id,,,,,
orfm.bz_0.1_98,617,11442,False,-,prodigal.bz_0.1_20
orfm.bz_10.1_101,616,9389,False,-,prodigal.bz_10.1_16
orfm.bz_10.1_67,625,5324,False,-,prodigal.bz_10.1_10
orfm.bz_9.1_140,619,12143,False,-,prodigal.bz_9.1_31
orfm.bz_8.1_128,622,10697,False,-,prodigal.bz_8.1_26
orfm.bz_7.1_89,638,9517,False,-,prodigal.bz_7.1_14
orfm.bz_5.1_95,610,8139,False,-,prodigal.bz_5.1_20
orfm.bz_11.1_140,618,11469,False,-,prodigal.bz_11.1_21
orfm.bz_4.1_203,598,1405,False,-,prodigal.bz_4.1_40


In [ ]:
genes_df.sort_values('length')[['length']]
df = pd.read_csv('../data/genes.csv', index_col=0)
df = df[(df.length > 300) & (df.length < 400)].copy()
df = df[df.num_tmhs == 4].copy()